In [1]:
import tensorflow as tf
from itertools import chain
from pathlib import Path
from tensorflow.core.example.feature_pb2 import BytesList, Int64List
from tensorflow.train import Example, Feature, Features
from tqdm.notebook import tqdm

POS = 1
NEG = 0
TRAIN_PATH = 'aclImdb/train'
TEST_PATH = 'aclImdb/test'
BATCH_SIZE = 32

def text_to_tfexample(label, text):
    feature = {
        'text': Feature(bytes_list=BytesList(value=[text.encode('utf-8')])),
        'label': Feature(int64_list=Int64List(value=[label]))
    }
    return Example(features=Features(feature=feature))

def read_directory(path, label):
    for path in Path(path).glob('*.txt'):
        text = path.read_text(encoding='utf-8')
        yield text_to_tfexample(text, label)

def load_set(base_path):
    set = chain(read_directory(base_path + '/pos', label=POS), read_directory(base_path + '/neg', label=NEG))
    return list(set)

def convert_reviews(directory, target):
    with tf.io.TFRecordWriter(target) as writer:
        pos_files = [(POS, p) for p in Path(f'{directory}/pos').iterdir() if p.is_file()]
        neg_files = [(NEG, p) for p in Path(f'{directory}/neg').iterdir() if p.is_file()]
        files = pos_files + neg_files
        for p in tqdm(files, desc="Reading files"):
            data = p[1].read_text(encoding='utf-8')
            writer.write(text_to_tfexample(p[0], data).SerializeToString())

# convert_reviews(TRAIN_PATH, 'train.tfrecord')
# convert_reviews(TEST_PATH, "test.tfrecord")

Reading files:   0%|          | 0/25000 [00:00<?, ?it/s]

In [24]:
def parse_tfrecord(example):
    feature_description = {
        'text': tf.io.FixedLenFeature([], tf.string),
        'label': tf.io.FixedLenFeature([], tf.int64)
    }
    parsed = tf.io.parse_single_example(example, feature_description)
    return (parsed['label'], parsed['text'])

def load_dataset(filename):
    train_ds = tf.data.TFRecordDataset(filename).map(lambda x: parse_tfrecord(x))
    X = []
    y = []
    for i in train_ds.as_numpy_iterator():
        X.append(i[1].decode("utf-8"))
        y.append(int(i[0]))
    return (X, y)

(X_train, y_train) = load_dataset('train.tfrecord')
(X_fulltest, y_fulltest) = load_dataset('test.tfrecord')
X_valid = X_fulltest[0:15_000]
y_valid = y_fulltest[0:15_000]

X_test = X_fulltest[15_000:]
y_test = y_fulltest[15_000:]
print(len(X_valid), len(X_test))
print(type(X_train[0]), type(y_train[0]), type(X_valid[0]), type(y_valid[0]))

15000 10000
<class 'str'> <class 'int'> <class 'str'> <class 'int'>


In [21]:
vectorize_layer = tf.keras.layers.TextVectorization(max_tokens=10_000, output_mode='int', output_sequence_length=250)
vectorize_layer.adapt(X_train)

print('Vocabulary size: ', len(vectorize_layer.get_vocabulary()))
vectorizer_model = tf.keras.Sequential([vectorize_layer])
vectorizer_model.save("text-vectorizer.keras")

Vocabulary size:  10000


In [25]:
import os
from keras.src.callbacks import EarlyStopping

loaded_vectorizer = tf.keras.models.load_model("text-vectorizer.keras").layers[0]

tf.random.set_seed(42)
model = tf.keras.Sequential([
    tf.keras.Input(dtype=tf.string, shape=(1,)),
    loaded_vectorizer,
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.compile(loss='binary_crossentropy', optimizer='sgd', metrics=['accuracy'])

early_stopping = EarlyStopping(
    monitor='val_loss',   # Metric to monitor
    patience=5,           # How many epochs to wait for improvement
    restore_best_weights=True # Keep the weights from the best epoch
)

root_logdir = os.path.join(os.curdir, "my_logs")

def get_run_logdir():
    import time
    run_id = time.strftime("run_%Y_%m_%d-%H_%M_%S")
    return os.path.join(root_logdir, run_id)

run_logdir = get_run_logdir()

tensorboard_cb = tf.keras.callbacks.TensorBoard(run_logdir)

history = model.fit(X_train, y_train, epochs=30, validation_data=(X_valid, y_valid), callbacks=[early_stopping, tensorboard_cb])

%load_ext tensorboard
%tensorboard --logdir=./my_logs --port=6006

Epoch 1/30
782/782 [==============================] - 4s 4ms/step - loss: 1866.3956 - accuracy: 0.4956 - val_loss: 0.6957 - val_accuracy: 0.1672
Epoch 2/30
782/782 [==============================] - 2s 2ms/step - loss: 5.7840 - accuracy: 0.4922 - val_loss: 0.6954 - val_accuracy: 0.1672
Epoch 3/30
782/782 [==============================] - 2s 2ms/step - loss: 0.6930 - accuracy: 0.4941 - val_loss: 0.6954 - val_accuracy: 0.1672
Epoch 4/30
782/782 [==============================] - 2s 2ms/step - loss: 0.6930 - accuracy: 0.4941 - val_loss: 0.6954 - val_accuracy: 0.1672
Epoch 5/30
782/782 [==============================] - 2s 3ms/step - loss: 0.6930 - accuracy: 0.4941 - val_loss: 0.6954 - val_accuracy: 0.1672
Epoch 6/30
782/782 [==============================] - 2s 2ms/step - loss: 0.6930 - accuracy: 0.4941 - val_loss: 0.6954 - val_accuracy: 0.1672
Epoch 7/30
782/782 [==============================] - 2s 2ms/step - loss: 0.6930 - accuracy: 0.4941 - val_loss: 0.6954 - val_accuracy: 0.1672
Epo